[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_block_course/blob/main/DAY4/session4b_student.ipynb)

# Session 4b — Galaxy Spectra Then and Now: Dimensionality Reduction of SDSS and JWST Spectra

In the concepts notebook we learned what PCA, t-SNE and UMAP preserve, on data whose structure we knew. Now we apply PCA and UMAP to real galaxy spectra from two very different epochs:

* **Part A — SDSS**, about 3000 galaxies at $0.02 < z < 0.1$, i.e. today. High signal-to-noise, resolution $R \approx 2000$.
* **Part B — JWST/NIRSpec PRISM**, about 1500 galaxies at $4 < z < 7$, less than 1.5 Gyr after the Big Bang, from the DAWN JWST Archive (DJA). Low resolution ($R \approx 30$–300) and much noisier.

We run the same pipeline on both and ask: **do galaxies at cosmic dawn vary in the same ways as galaxies today? And which of them does our description fail?**

The data were prepared with `prepare_sdss_spectra.py` and `prepare_dja_spectra.py`: every spectrum is shifted to the rest frame, put on a common wavelength grid and normalised. The methods never see the catalogue measurements (line ratios, stellar masses, ...); we only use them to interpret the results.

In [ ]:
import warnings
for msg in ["n_jobs value", "IProgress not found", "The default value of `copy`"]:
    warnings.filterwarnings("ignore", message=msg)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap

SDSS_DIR, JWST_DIR = Path("DATA/sdss_data"), Path("DATA/dja_data")

GROUP_COLORS = {"star-forming": "C0", "composite": "C2", "AGN": "C1", "no lines": "gray"}
LINES_SDSS = {"[OII]": 3727, "CaII K": 3934, "CaII H": 3969, r"H$\delta$": 4102, r"H$\beta$": 4861, "[OIII]": 5007,
              "Mg b": 5175, "Na D": 5893, r"H$\alpha$": 6563, "[SII]": 6724}
LINES_JWST = {"CIV": 1549, "CIII]": 1909, "MgII": 2799, "[OII]": 3727, "Balmer limit": 3646,
              "[NeIII]": 3869, r"H$\gamma$": 4340, r"H$\beta$": 4861, "[OIII]": 5007, r"H$\alpha$": 6563}

plt.rcParams.update({
    "figure.figsize": (7, 5), "figure.dpi": 110, "font.size": 12, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 1.2, "legend.frameon": False,
})

def mark_lines(ax, lines=LINES_SDSS, ymin=0.97):
    x0, x1 = ax.get_xlim()
    last = -np.inf
    for name, w in sorted(lines.items(), key=lambda kv: kv[1]):
        if x0 < w < x1:
            ax.axvline(w, color="gray", lw=0.6, ls=":", zorder=0)
            close = (w - last) < 0.02*(x1 - x0)          # put the label of a close neighbour on the other side
            ax.text(w, ymin, name, transform=ax.get_xaxis_transform(), rotation=90, fontsize=8,
                    ha="left" if close else "right", va="top", color="gray")
            last = w

def group_scatter(ax, groups, Z, s=4):
    """Scatter 2D points Z, coloured by a categorical BPT group."""
    for grp, col in GROUP_COLORS.items():
        m = (groups == grp).to_numpy()
        ax.scatter(*Z[m].T, s=s, color=col, label=grp, linewidth=0)

def colored_scatter(ax, fig, Z, c, cmap="viridis", vmin=None, vmax=None, s=4, cbar_label=None, **kwargs):
    """Scatter 2D points Z, coloured by a continuous quantity c, with an optional colorbar."""
    sc = ax.scatter(*Z.T, s=s, c=c, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0)
    if cbar_label is not None:
        fig.colorbar(sc, ax=ax, label=cbar_label, shrink=0.8)
    ax.set(**kwargs)
    return sc

In [ ]:
from pathlib import Path

# Downloads the SDSS/JWST spectra bundle from Google Drive into DATA/ on first run.
if not Path("DATA").exists():
    import subprocess, zipfile
    subprocess.run(["pip", "install", "-q", "gdown"], check=True)
    import gdown
    gdown.download(id="1Wh218Vdo2V2xaPtHsLhcK7jQE9Wwlguq", output="session4b_data.zip", quiet=False)
    with zipfile.ZipFile("session4b_data.zip") as z:
        z.extractall(".")

# Part A — SDSS galaxies today

The SDSS spectra cover 3750–7000 Å in the rest frame on 1000 wavelengths, in $f_\lambda$, normalised to the median flux at 5300–5700 Å. For each galaxy we also have measurements from the **MPA-JHU** catalogue (Brinchmann et al. 2004; Kauffmann et al. 2003):

* the **BPT class** from emission-line ratios: star-forming, composite, AGN, or no measurable lines,
* $D_n4000$, the strength of the 4000 Å break — a measure of the age of the stellar population,
* stellar mass and specific star formation rate.

In [ ]:
d = np.load(SDSS_DIR / "sdss_spectra.npz")
wave, F = d["wave"], d["flux"].astype(float)
cat = pd.read_csv(SDSS_DIR / "sdss_catalog.csv")

# MPA-JHU BPT classes: 1 SF, 2 low-S/N SF, 3 composite, 4 AGN, 5 low-S/N LINER, -1 not classifiable
cat["group"] = cat.bptclass.map({1: "star-forming", 2: "star-forming", 3: "composite",
                                 4: "AGN", 5: "AGN", -1: "no lines"}).fillna("no lines")
cat["log_ssfr"] = cat.specsfr_tot_p50
print(f"{F.shape[0]} spectra x {F.shape[1]} wavelengths")
print(cat.group.value_counts())

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(3750, 7700)
for k, (grp, col) in enumerate(GROUP_COLORS.items()):
    for i in np.where(cat.group == grp)[0][:2]:
        ax.plot(wave, F[i] + 1.3*k, color=col)
    ax.text(7050, 1.3*k + 1, grp, color=col, va="center")
mark_lines(ax)
ax.set(xlabel="rest-frame wavelength [Å]", ylabel="normalised flux + offset",
       title="SDSS: two example spectra per BPT group");

## A1. How many dimensions do galaxy spectra have?

Each spectrum is a point in a 1000-dimensional space. In the concepts notebook, the eigenvalue spectrum of PCA showed the number of dimensions that carry structure, followed by a flat floor set by the noise.

We estimate the noise per pixel from the scatter between neighbouring pixels (the spectra are smooth on the scale of one pixel, so the difference between neighbours is mostly noise). We do **not** standardise: every pixel has the same units, and standardising would inflate the pixels that are mostly noise.

*Why is the noise floor above $\sigma^2$?* With $N$ spectra and $D$ pixels, even pure noise does not give $D$ equal eigenvalues: the sample eigenvalues spread out, up to about $\sigma^2(1 + \sqrt{D/N})^2$ (the Marchenko–Pastur law). Components below this line cannot be distinguished from noise.

### ✏️ Exercise A1 — The eigenvalue spectrum

**Predict first:** how many components do you think you need to describe galaxy spectra down to the noise — 2? 10? 100?

1. Fit `PCA(n_components=100)` to `F` and store it as `pca`.
2. How many components lie clearly above the noise? How does the spectrum differ from the clean "signal + flat floor" of the simulated data, and why?
3. Choose the number of components `N_PC` you will keep.

In [ ]:
noise_i = 1.4826*np.median(np.abs(np.diff(F, axis=1)), axis=1) / np.sqrt(2)   # noise per pixel, per spectrum
noise_var = np.mean(noise_i**2)
mp_edge = noise_var * (1 + np.sqrt(F.shape[1] / F.shape[0]))**2                 # largest pure-noise eigenvalue

# YOUR CODE HERE
# pca = 
# N_PC = 

lam = pca.explained_variance_
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
ax1.semilogy(np.arange(1, 101), lam, "o-", ms=3, color="C0")
ax1.axhline(noise_var, color="C1", ls="--", label=r"noise variance $\sigma^2$")
ax1.axhline(mp_edge, color="C1", ls=":", label="largest eigenvalue of pure noise")
ax1.axvline(N_PC, color="black", lw=1, ls=":")
ax1.set(xlabel="component", ylabel="eigenvalue", title="SDSS: eigenvalue spectrum", xscale="log")
ax1.legend()
ax2.plot(np.arange(1, 101), np.cumsum(pca.explained_variance_ratio_), color="C0")
ax2.axvline(N_PC, color="black", lw=1, ls=":")
ax2.set(xlabel="component", ylabel="cumulative explained variance", xscale="log", ylim=(0, 1.01),
        title=f"{N_PC} components: {np.sum(pca.explained_variance_ratio_[:N_PC]):.1%} of the variance")
fig.tight_layout();

**Discussion**

* The eigenvalues never cross the "largest eigenvalue of pure noise" line, even at $k = 100$ — unlike the clean "signal + flat floor" of the simulated data. `noise_var` here only captures *uncorrelated* pixel-to-pixel noise (photon/read noise); it misses correlated systematics that affect many pixels at once (flux-calibration residuals, sky-subtraction residuals, template-mismatch in the normalisation). Those show up as extra, small, *correlated* eigenvalues that sit above the naive noise line but aren't galaxy physics either. On top of that, real galaxies vary continuously in SFH, metallicity, dust and velocity dispersion — genuinely high-dimensional, unlike the three structures in the simulated data. So the Marchenko–Pastur edge is a *lower bound* on the real noise floor, not the true one, and the spectrum decays smoothly instead of hitting a knee.

| $k$ | eigenvalue | $\lambda$ / noise-edge | cumulative variance |
|---|---|---|---|
| 5   | 0.350  | 91×  | 90.5% |
| 10  | 0.110  | 29×  | 93.5% |
| 20  | 0.018  | 4.8× | 94.9% |
| 30  | 0.0084 | 2.2× | 95.2% |
| 50  | 0.0065 | 1.7× | 95.7% |
| 80  | 0.0050 | 1.3× | 96.2% |
| 100 | 0.0042 | 1.1× | 96.5% |

* **So how do you pick `N_PC` without a clean cutoff?** Not from "components above the noise line" — pick based on what the components are *for*:
  1. **Cumulative variance threshold**: 90% is reached by 5 components, 95% by about 20; each additional component after that buys very little (80 more components only adds 1.3 percentage points).
  2. **Where the decay rate itself flattens**: the log-log slope of the eigenvalue spectrum steepens sharply up to $k \approx 10$, then levels off — the closest thing to an "elbow" here.
  3. **Downstream stability** (the most rigorous check): vary `N_PC` and see whether what you actually use it for — the UMAP map's shape, the B6 outlier list — stops changing. If 15 and 30 components give the same answer, you've kept enough.

  `N_PC = 20` satisfies (1) and (2): it captures 95% of the variance and sits right where the spectrum's decay visibly bends.

## A2. What do the eigenspectra show?

Each principal component is itself a spectrum — an **eigenspectrum**. Every galaxy is the mean spectrum plus a weighted sum of eigenspectra, and the weights (coefficients) are its new, compact description.

### ✏️ Exercise A2 — Read the eigenspectra

1. Plot the mean spectrum and the first four eigenspectra (`pca.mean_`, `pca.components_[j]`) using `mark_lines`.
2. For each of the first three components, describe what adding it with a **positive** coefficient does to a spectrum. Which one tracks the age of the stellar population, and which ones the emission lines?
3. Plot the first two coefficients of each galaxy (`A = pca.transform(F)`), coloured once by BPT group and once by $D_n4000$ (`cat.d4000_n`). Does this match your interpretation?

*Note:* the sign of a principal component is arbitrary — `-e` is as good an eigenspectrum as `e`.

In [ ]:
# YOUR CODE HERE
fig, axes = plt.subplots(11, 1, figsize=(12, 22), sharex=True)
#axes[0].plot(wave, ... , color="black")
axes[0].set_ylabel("mean")
for j in range(10):
#    axes[j + 1].plot(wave, ... , color="C0")
    axes[j + 1].axhline(0, color="gray", lw=0.8)
    axes[j + 1].set_ylabel(f"e{j + 1}\n({pca.explained_variance_ratio_[j]:.1%})")
for ax in axes:
    mark_lines(ax)
axes[-1].set_xlabel("rest-frame wavelength [Å]")
fig.tight_layout()



In [ ]:
A = pca.transform(F)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
group_scatter(ax1, cat.group, A[:, :2])
ax1.set(xlabel="a1", ylabel="a2", title="first two PCA coefficients, by BPT group")
ax1.legend(markerscale=4)
colored_scatter(ax2, fig, A[:, :2], cat.d4000_n, vmin=1.0, vmax=2.0,
                cbar_label=r"$D_n4000$", xlabel="a1", ylabel="a2", title=r"coloured by $D_n4000$")
fig.tight_layout()

## A3. A UMAP map of SDSS spectra

As in the concepts notebook, we run UMAP not on the 1000 raw pixels but on the first `N_PC` PCA coefficients. This removes the noise-dominated dimensions first, and makes UMAP much faster.

### ✏️ Exercise A3 — Colour the map by physics

1. Make a UMAP map `Z` of the first `N_PC` PCA coefficients (`umap.UMAP(n_neighbors=15, random_state=0)`).
2. The first panel always colours by BPT group. For the other three panels, **choose your own** columns from `cat.columns` (e.g. `d4000_n`, `lgm_tot_p50`, `log_ssfr`, `h_alpha_flux`, `oiii_5007_flux`, `nii_6584_flux`, `bptclass`, ...) and reasonable `(vmin, vmax)` ranges for each.

**Before looking:** for each column you pick, predict whether it will vary *smoothly* across the map or pick out separate regions — then check. Remember what the concepts notebook taught about **distances and sizes** in the map before you interpret the gaps between regions.

*Hint:* `colored_scatter(ax, fig, Z, cat[col], vmin=..., vmax=..., cbar_label=label, title=label, xticks=[], yticks=[])` draws one panel; `cat[col].describe()` is a quick way to pick sensible `vmin`/`vmax`.

In [ ]:
# YOUR CODE HERE


fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
group_scatter(axes[0], cat.group, Z, s=3)
axes[0].legend(markerscale=4, fontsize=9)
axes[0].set(title="BPT group", xticks=[], yticks=[])
for ax, (col, label, lims) in zip(axes[1:], panels):
    colored_scatter(ax, fig, Z, cat[col], vmin=lims[0], vmax=lims[1], s=3,
                     cbar_label=label, title=label, xticks=[], yticks=[])
fig.tight_layout()

# Part B — JWST galaxies at cosmic dawn

The **DAWN JWST Archive** (DJA) has reduced, in a uniform way, almost all public NIRSpec multi-object spectroscopy, and visually inspected the redshifts ([merged table v4](https://dawn-cph.github.io/dja/blog/2025/05/01/nirspec-merged-table-v4/); de Graaff et al. 2024, Heintz et al. 2024). We use **PRISM** spectra (0.6–5.3 µm) with a robust redshift (grade 3) at $4 < z < 7$. At these redshifts every spectrum covers the same rest-frame range, **1300–6700 Å**: the UV continuum, the Balmer break, H$\beta$ + [OIII] and H$\alpha$.

Differences from SDSS that matter for what follows:

* the spectra are in $f_\nu$ (µJy), normalised to their median — a young, blue galaxy has a *flat* UV continuum in $f_\nu$;
* the grid has only 250 wavelengths, because PRISM has low resolution — lines are unresolved and appear wider than in SDSS;
* every spectrum comes with its own **error spectrum** `EJ`, and the noise is much larger and varies a lot from galaxy to galaxy;
* the sample is a mixture of many programmes, each with its own target selection — it is **not** a representative sample of galaxies at $4<z<7$.

For interpretation we have, from the DJA table: the redshift, the rest-frame equivalent widths of [OIII] and H$\alpha$, and the stellar mass and dust attenuation $A_V$ from fits to the photometry.

In [ ]:
dj = np.load(JWST_DIR / "jwst_spectra.npz")
wave_j, FJ, EJ, bad_j = dj["wave"], dj["flux"].astype(float), dj["err"].astype(float), dj["bad"]
catj = pd.read_csv(JWST_DIR / "jwst_catalog.csv")
catj["survey"] = catj["root"].str.split("-").str[0]
snr_j = np.median(FJ / EJ, axis=1)                           # median S/N per pixel
print(f"{FJ.shape[0]} spectra x {FJ.shape[1]} wavelengths,  {catj.z.min():.2f} < z < {catj.z.max():.2f}")
print(catj.survey.value_counts().head(8).to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw=dict(width_ratios=[2.2, 1]))
ax1.set_xlim(1300, 7400)
examples = [np.argmin(np.abs(catj.z - zz) + 10*(snr_j < np.percentile(snr_j, 60))) for zz in (4.3, 5.2, 6.1, 6.8)]
for k, i in enumerate(examples):
    ax1.plot(wave_j, FJ[i] + 2.5*k, color="C0", drawstyle="steps-mid")
    ax1.fill_between(wave_j, FJ[i] - EJ[i] + 2.5*k, FJ[i] + EJ[i] + 2.5*k, color="C0", alpha=0.25, step="mid", lw=0)
    ax1.text(6800, 2.5*k + 1, f"z = {catj.z[i]:.2f}", color="C0", va="center")
mark_lines(ax1, LINES_JWST)
ax1.set(xlabel="rest-frame wavelength [Å]", ylabel=r"normalised $f_\nu$ + offset", title="JWST/NIRSpec PRISM: example spectra")
#ax2.hist(1/noise_i, bins=np.geomspace(0.5, 300, 50), color="C0", alpha=0.7, label="SDSS")
ax2.hist(snr_j, bins=np.geomspace(0.5, 300, 50), color="C1", alpha=0.7, label="JWST PRISM")
ax2.set(xscale="log", xlabel="median S/N per pixel", ylabel="spectra", title="Signal-to-noise")
ax2.legend()
fig.tight_layout();

## B4. Do galaxies at cosmic dawn vary in the same way as galaxies today?

### ✏️ Exercise B4 — PCA of JWST spectra, compared with SDSS

**Predict first:** will the JWST spectra need *more* or *fewer* components than SDSS above the noise? Which features do you expect in the first eigenspectra?

1. Fit `PCA(n_components=50)` to `FJ` and store it as `pca_j`. Choose `N_PC_J`, the number of components you keep.
2. The first plot shows both eigenvalue spectra **in units of each sample's noise variance**, so that "1" means "as large as the noise". How many JWST components lie above the noise edge? Does that mean that galaxies at $z \sim 5$ are *simpler* than today's — or what else could it mean?
3. The second plot shows the JWST mean spectrum (with the SDSS mean converted to $f_\nu$ for comparison) and the first three eigenspectra. Describe each component. Which features carry the variance at $z\sim5$, and how does this compare with SDSS eigenspectra from Exercise A2?

In [ ]:
noise_var_j = np.mean(EJ[~bad_j]**2)
mp_edge_j = noise_var_j * (1 + np.sqrt(FJ.shape[1] / FJ.shape[0]))**2

# YOUR CODE HERE


fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(np.arange(1, 101), pca.explained_variance_ / noise_var, "o-", ms=3, color="C0", label="SDSS")
ax.loglog(np.arange(1, 51), pca_j.explained_variance_ / noise_var_j, "o-", ms=3, color="C1", label="JWST PRISM")
ax.axhline(mp_edge / noise_var, color="C0", ls=":")
ax.axhline(mp_edge_j / noise_var_j, color="C1", ls=":", label="noise edge (Marchenko–Pastur)")
ax.axvline(N_PC_J, color="C1", lw=1, ls="--")
ax.set(xlabel="component", ylabel="eigenvalue / noise variance", title="Eigenvalues relative to the noise")
ax.legend()

sdss_mean_fnu = pca.mean_ * wave**2                          # f_nu ∝ f_lambda λ²
sdss_mean_fnu /= np.median(sdss_mean_fnu)
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
axes[0].plot(wave_j, pca_j.mean_, color="C1", drawstyle="steps-mid", label="JWST, 4 < z < 7")
axes[0].plot(wave, sdss_mean_fnu * np.median(pca_j.mean_[(wave_j > 3750)]) / np.median(sdss_mean_fnu),
             color="C0", lw=1, label=r"SDSS, z ≈ 0.05 (in $f_\nu$)")
axes[0].set_ylabel("mean")
axes[0].legend(loc="center left", fontsize=9)
for j in range(3):
    axes[j + 1].plot(wave_j, pca_j.components_[j], color="C1", drawstyle="steps-mid")
    axes[j + 1].axhline(0, color="gray", lw=0.8)
    axes[j + 1].set_ylabel(f"e{j + 1}\n({pca_j.explained_variance_ratio_[j]:.0%})")
for ax in axes:
    ax.set_xlim(1300, 7000)
    mark_lines(ax, LINES_JWST)
axes[-1].set_xlabel("rest-frame wavelength [Å]")
fig.tight_layout();

## B5. A UMAP map of JWST spectra

### ✏️ Exercise B5 — What organises the map?

1. Make a UMAP map `ZJ` of the first `N_PC_J` coefficients of `pca_j` (`n_neighbors=15, random_state=0`). Store the coefficients in `AJ`.
2. The cell colours the map by redshift, [OIII] equivalent width, stellar mass, $A_V$ and survey.

**Before looking:** within $4 < z < 7$, should the map be organised by redshift? After looking: if redshift *does* vary across the map, list possible reasons. Think about the galaxies (evolution over ~1 Gyr), the instrument (which observed wavelength — and so which resolution and noise — a given rest-frame wavelength falls at), and the sample (which programmes target which redshifts).

In [ ]:
catj.columns

In [ ]:
# YOUR CODE HERE
# AJ 
# ZJ 

# pick a few panels to display

# fig, axes = plt.subplots(1, 5, figsize=(24, 4.6))
# panels = [("z", "redshift", (4, 7), "viridis"),
#           ("line_ha", "log EW([OIII]), rest frame [Å]", (1.5, 3.3), "magma"),
#           ("phot_mass", r"log $M_*/M_\odot$", (7.5, 10.5), "viridis"),
#           ("phot_Av", r"$A_V$", (0, 1.5), "viridis")]
# for ax, (col, label, lims, cmap) in zip(axes, panels):
#     val = np.log10(catj[col].clip(lower=1)) if col.startswith("ew") else catj[col]
#     colored_scatter(ax, fig, ZJ, val, cmap=cmap, vmin=lims[0], vmax=lims[1], s=5,
#                      cbar_label=label, title=label, xticks=[], yticks=[])

# top = catj.survey.value_counts().index[:5]
# palette = ["C0", "C1", "C2", "C8", "C6"]
# axes[4].scatter(*ZJ.T, s=4, color="lightgray", linewidth=0, label="other")
# for survey, col in zip(top, palette):
#     m = (catj.survey == survey).to_numpy()
#     axes[4].scatter(*ZJ[m].T, s=5, color=col, linewidth=0, label=survey)
# axes[4].legend(markerscale=3, fontsize=8)
# axes[4].set(title="survey", xticks=[], yticks=[])
# fig.tight_layout()

### ✏️ Robustness check — does a structure survive?

Pick one structure you noticed in the map above — a cluster, a gap, a gradient in one of the panels. Before trusting it, recall the lesson from the concepts notebook: check that it survives a change of `n_neighbors` and of the random seed.

Re-run UMAP on `AJ` with different settings, and compare the two maps coloured by the same property. Does what you noticed survive?

In [ ]:
# YOUR CODE HERE
# ZJ_alt = umap.UMAP(n_neighbors=5, random_state=1).fit_transform(AJ)
# color_col, color_label, lims = "z", "redshift", (4, 7)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colored_scatter(ax1, fig, ZJ, catj[color_col], vmin=lims[0], vmax=lims[1],
                 cbar_label=color_label, title="original map", xticks=[], yticks=[])
colored_scatter(ax2, fig, ZJ_alt, catj[color_col], vmin=lims[0], vmax=lims[1],
                 cbar_label=color_label, title="different n_neighbors / seed", xticks=[], yticks=[])
fig.tight_layout()

## B6. Which galaxies does PCA fail to describe?

If `N_PC_J` components describe a typical galaxy down to the noise, then a galaxy that is poorly reconstructed is **unusual**. Finding the objects that do not fit is one of the most useful applications of PCA in large surveys. JWST has found several kinds of galaxies that were not expected — let's see whether PCA finds them without being told.

Because we have an error spectrum, we can measure the residual in units of the noise:
$$
\text{score}_i = \sqrt{\frac{1}{N_\text{good}} \sum_\lambda \left(\frac{F_{i\lambda} - \hat F_{i\lambda}}{E_{i\lambda}}\right)^2}
$$
A score of 1 means "reconstructed down to the noise".

### ✏️ Exercise B6 — Outliers

1. Reconstruct every spectrum from the first `N_PC_J` components (`PCA(n_components=N_PC_J)`, `.transform`, `.inverse_transform`) into `FJ_rec`, and compute `score` as above, ignoring the bad pixels (`bad_j`; `np.where` and `np.nanmean` help).
2. The cell shows the eight worst-fit spectra. What are they? Look for:
   * a **V-shaped** continuum — blue in the UV, turning red at the Balmer limit — often with **broad** H$\alpha$/H$\beta$: the "little red dots" (Matthee et al. 2024; Kocevski et al. 2025);
   * a strong **Balmer break** with weak lines: galaxies that have (temporarily) stopped forming stars;
   * extreme emission lines, damped Lyman-$\alpha$ absorption at the blue end, or problems with the data.
3. The median score of normal galaxies is not exactly 1. What does a median above (or below) 1 tell you about the error spectra?
4. The second plot shows the score against S/N. Why do bright galaxies have higher scores? What does this mean for a list of "the most unusual galaxies", and how could you correct for it?
5. Where do the outliers lie in the UMAP map? Are they on their own island?

In [ ]:
# YOUR CODE HERE
# pca_jk = PCA(n_components=N_PC_J).fit(FJ)
# FJ_rec = pca_jk.inverse_transform(pca_jk.transform(FJ))
# chi = np.where(bad_j, np.nan, (FJ - FJ_rec) / EJ)
# score = np.sqrt(np.nanmean(chi**2, axis=1))

catj["pca_score"] = score
worst = np.argsort(score)[::-1][:10]
print(f"median score: {np.median(score):.2f}")

fig, axes = plt.subplots(5, 2, figsize=(15, 12), sharex=True)
for ax, i in zip(axes.flat, worst):
    ax.fill_between(wave_j, FJ[i] - EJ[i], FJ[i] + EJ[i], color="gray", alpha=0.3, step="mid", lw=0)
    ax.plot(wave_j, FJ[i], color="black", lw=0.8, drawstyle="steps-mid", label="data")
    ax.plot(wave_j, FJ_rec[i], color="C1", lw=1.2, label=f"{N_PC_J}-component reconstruction")
    ax.set_xlim(1300, 6700)
    mark_lines(ax, LINES_JWST)
    ax.set_title(f"{catj.file[i].replace('.spec.fits', '')}\nz = {catj.z[i]:.2f}, S/N {snr_j[i]:.0f}, score {score[i]:.1f}",
                 fontsize=9)
axes[0, 0].legend(fontsize=8, loc="center left")
for ax in axes[-1]:
    ax.set_xlabel("rest-frame wavelength [Å]")
fig.tight_layout()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
ax1.scatter(snr_j, score, s=5, color="C0", linewidth=0)
ax1.scatter(snr_j[worst], score[worst], s=40, facecolor="none", edgecolor="C1", label="worst 10")
ax1.set(xscale="log", yscale="log", xlabel="median S/N per pixel", ylabel="score", title="Score vs. signal-to-noise")
ax1.legend()
colored_scatter(ax2, fig, ZJ, np.log10(score), vmin=0, vmax=1, s=5, cbar_label="log score",
                title="UMAP map coloured by log score", xticks=[], yticks=[])
ax2.scatter(*ZJ[worst].T, s=60, marker="x", color="C1", label="worst 10")
ax2.legend()
fig.tight_layout()

*Tip:* you can look up any of these spectra in the DJA table viewer by searching for the file name; it shows the full 2D and 1D spectrum and the photometry.

Here: https://s3.amazonaws.com/msaexp-nirspec/extractions/nirspec_public_v4.4.html

### ✏️ Exercise B6b — Find your own outliers *(open-ended)*

The worst-10 list above is dominated by bright objects, because `score` grows with S/N. Try one of these (or your own idea):

* Restrict to a narrow S/N range (e.g. `snr_j < 20`) before ranking by `score`, to find faint unusual galaxies that the global ranking misses.
* Restrict to one `survey` (`catj.survey`) or one redshift range, and ask whether the "unusual" objects there are the same *kind* of outlier as the global worst 10.
* Define your own score — e.g. compute it only over a narrow wavelength range (the Balmer-break region, say) instead of the whole spectrum.

Make a short list of candidates, plot a few with their reconstruction (reuse the pattern from the cell above), and look at least one up in the DJA table viewer. What did you find — a known type of object, a new kind of outlier, or a data problem?

In [ ]:
# YOUR CODE HERE
# subset = ...                                             # restrict to faint/noisy spectra, high z, low z
# rank = np.where(subset, score, -np.inf)                  # push excluded objects to the bottom
# worst_faint = np.argsort(rank)[::-1][:6]

fig, axes = plt.subplots(3, 2, figsize=(15, 8), sharex=True)
for ax, i in zip(axes.flat, worst_faint):
    ax.fill_between(wave_j, FJ[i] - EJ[i], FJ[i] + EJ[i], color="gray", alpha=0.3, step="mid", lw=0)
    ax.plot(wave_j, FJ[i], color="black", lw=0.8, drawstyle="steps-mid", label="data")
    ax.plot(wave_j, FJ_rec[i], color="C1", lw=1.2, label=f"{N_PC_J}-component reconstruction")
    ax.set_xlim(1300, 6700)
    mark_lines(ax, LINES_JWST)
    ax.set_title(f"{catj.file[i].replace('.spec.fits', '')}\nz = {catj.z[i]:.2f}, S/N {snr_j[i]:.0f}, score {score[i]:.1f}",
                 fontsize=9)
axes[0, 0].legend(fontsize=8, loc="center left")
for ax in axes[-1]:
    ax.set_xlabel("rest-frame wavelength [Å]")
fig.tight_layout()

## Bonus — engineer your own feature *(optional)*

Pick or compute one simple quantity from the spectra that isn't already a PCA/UMAP axis, and check where it lands in both. A good starting point: the strength of the Balmer break, using two line-free continuum windows around the Balmer limit at rest 3646 Å (marked on earlier spectral plots).

1. Compute `continuum_blue` and `continuum_red` as the median flux in two narrow rest-frame windows, one just blueward and one just redward of 3646 Å — avoiding any marked emission lines.
2. Form a ratio (or difference) that captures the break strength.
3. Colour the PCA-coefficient plot (`AJ[:, 0]`, `AJ[:, 1]`) and the UMAP map `ZJ` by your feature. Is it aligned with one PCA component, or spread across several (recall A2's note that components aren't physical templates)? Does it vary smoothly across the UMAP map, or mark out a region (e.g. the Balmer-break outliers from B6)?

*Hint:* `wave_j` is the rest-frame grid; use a boolean mask like `(wave_j > 3500) & (wave_j < 3600)` to select a window, then `np.median(FJ[:, window], axis=1)`.

In [ ]:
# YOUR CODE HERE


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colored_scatter(ax1, fig, AJ[:, :2], break_strength, cbar_label=feature_label,
                 xlabel="a1", ylabel="a2", title="PCA coefficients")
colored_scatter(ax2, fig, ZJ, break_strength, cbar_label=feature_label,
                 title="UMAP map", xticks=[], yticks=[])
fig.tight_layout()